# Étape 15 — Les données des élèves permettent-elles de retrouver les prérequis experts ?

Jusqu'ici, le graphe était **donné** au modèle (expert, LLM, ou contrôles). Beaucoup d'articles proposent au contraire d'**apprendre** le graphe à partir des réponses des élèves,
et affirment que les flèches apprises sont « interprétables », c'est-à-dire proches des vrais prérequis. On le teste directement.

**Le modèle.** PrereqKT, mais chaque flèche possible A → B entre deux exercices du **même thème** (22 760 paires, les mêmes que celles évaluées par le LLM) reçoit un **poids appris** entre 0 et 1.
Le modèle apprend en même temps à prédire les réponses et les poids des flèches, avec :
- une pénalité de **parcimonie** (L1) qui pousse les poids inutiles vers 0 ;
- une pénalité d'**acyclicité** (NOTEARS, Zheng et al., 2018) qui décourage les cycles A → B → A.

Deux départs :
- `sans_prior` : toutes les flèches démarrent au même poids (0,1) ; tout ce qui est appris vient des données ;
- `prior_llm` : chaque flèche démarre au score du LLM (P(oui) de l'étape 9) ; les données peuvent-elles **corriger** le LLM ?

**Ce qu'on mesure.**
1. **Retrouve-t-on les prérequis experts ?** On classe les 22 760 paires par poids appris et on calcule l'**AUC** face au graphe expert (0,5 = hasard), et la précision sur les K premières paires (K = nombre de flèches expertes parmi les candidates).
2. **Est-ce seulement l'ordre du programme ?** Dans Junyi, beaucoup d'exercices sont faits dans un ordre imposé. Un modèle peut donc « retrouver » A → B simplement parce que A vient toujours avant B.
   On compare à une **référence naïve** : *la part des élèves qui font A avant B*. Et on sépare les paires à **ordre imposé** (≥ 95 % des élèves dans le même ordre) des paires à **ordre variable**.
3. **Le graphe appris aide-t-il le KT ?** AUC des premières tentatives, comparée à `aucun` et `expert`.

**Plan :** 2 départs × 2 régimes (**100 %** et **5 %** des élèves) × **5 graines** = 20 entraînements, plus `aucun` et `expert` à 100 % (10 entraînements ; à 5 %, on reprend l'étape 13).

**Hypothèses déclarées avant l'expérience** (tests t unilatéraux par graine, Holm sur les 8 tests) :
- **H5** — les données contiennent un signal : AUC(`sans_prior`) > 0,5 ;
- **H6** — l'apprentissage fait mieux que l'ordre de passage : AUC(`sans_prior`) > AUC(ordre de passage) ;
- **H7** — les données corrigent le LLM : AUC(`prior_llm`) > AUC(score LLM de départ) ;
- **H8** — au-delà de l'ordre imposé : AUC(`sans_prior`) > 0,5 sur les paires à **ordre variable**.

### Deux comptes en parallèle
`PARTIE = 'A'` (graines 0, 1, 2) et `PARTIE = 'B'` (graines 3, 4) écrivent chacune leur propre fichier : vous pouvez les lancer **en même temps sur deux comptes**.
Commencez par la cellule L2 (test rapide) : elle donne le temps par époque.

**GPU T4 obligatoire.**

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule L1 — Paires candidates (même thème), score du LLM, vrais prérequis, ordre de passage des élèves
from collections import Counter
from itertools import permutations

scores = pd.read_csv(os.path.join(RESULTATS, 'junyi_llm_scores.csv'))
scores[['A', 'B']] = scores[['A', 'B']].astype(int)
groupes = exos.dropna(subset=['topic']).groupby('topic')['exo_id'].apply(list)
cand = pd.DataFrame([(a, b) for ids in groupes for a in ids for b in ids if a != b], columns=['A', 'B'])
cand = cand.merge(scores[['A', 'B', 'p_oui']].drop_duplicates(['A', 'B']), on=['A', 'B'], how='left')
manque = int(cand.p_oui.isna().sum())
if manque:
    print(f'Attention : {manque} paires sans score LLM, remplacées par la médiane')
    cand['p_oui'] = cand.p_oui.fillna(scores.p_oui.median())
PAIRES = cand[['A', 'B']].values.astype(np.int64)

G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
anc = {b: nx.ancestors(G_exp, b) for b in range(N)}
cand['expert'] = A_expert[cand.A, cand.B] > 0
cand['ancetre'] = [a in anc[b] for a, b in PAIRES]
topic_de = exos.set_index('exo_id')['topic'].to_dict()

def eleves_choisis(fraction, graine):          # même tirage que sous_ensemble_train
    u = donnees['train'][0]
    if fraction >= 1:
        return set(u)
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    return set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))

def ordre_passage(fraction, graine):
    # Pour chaque paire (A, B) : nombre d'élèves ayant fait A et B, et nombre de ceux qui ont fait A en premier (1re tentative)
    u, q, _, f = donnees['train']
    gardes = eleves_choisis(fraction, graine)
    rang, compteur = {}, {}
    for uid, qq, ff in zip(u, q, f):
        if uid not in gardes:
            continue
        d = rang.setdefault(uid, {}); k = compteur.get(uid, 0)
        for x, fl in zip(qq, ff):
            if fl:
                d[int(x)] = k
            k += 1
        compteur[uid] = k
    deux, avant = Counter(), Counter()
    for d in rang.values():
        par_theme = {}
        for x in d:
            t = topic_de.get(x)
            if isinstance(t, str):
                par_theme.setdefault(t, []).append(x)
        for xs in par_theme.values():
            for a, b in permutations(xs, 2):
                deux[(a, b)] += 1
                if d[a] < d[b]:
                    avant[(a, b)] += 1
    n2 = np.array([deux[(a, b)] for a, b in PAIRES]); na = np.array([avant[(a, b)] for a, b in PAIRES])
    return n2, na

n2, na = ordre_passage(1.0, 0)                 # description du programme : tous les élèves d'entraînement
part_avant = na / np.maximum(n2, 1)
cand['n_eleves_deux'] = n2
cand['ordre'] = np.where(n2 < 20, 'rare', np.where((part_avant >= 0.95) | (part_avant <= 0.05), 'impose', 'variable'))
SOUS_ENS = {'toutes': np.ones(len(cand), bool), 'ordre_impose': (cand.ordre == 'impose').values,
            'ordre_variable': (cand.ordre == 'variable').values}

print(f'{len(cand):,} paires candidates | flèches expertes parmi elles : {int(cand.expert.sum())} '
      f'| prérequis directs ou indirects : {int(cand.ancetre.sum())}')
print(cand.groupby('ordre').agg(paires=('A', 'size'), dont_expertes=('expert', 'sum')).to_string())

def qualite(score, prefixe):
    # AUC face aux flèches expertes (directes), AUC face aux prérequis directs ou indirects, précision@K, par sous-ensemble
    score = np.asarray(score, dtype=float); res = {}
    for nom, m in SOUS_ENS.items():
        y, ya, s = cand.expert.values[m], cand.ancetre.values[m], score[m]
        ok = len(set(y)) == 2
        res[f'{prefixe}AUC_{nom}'] = round(roc_auc_score(y, s), 4) if ok else None
        res[f'{prefixe}AUC_etendue_{nom}'] = round(roc_auc_score(ya, s), 4) if len(set(ya)) == 2 else None
        K = int(y.sum())
        res[f'{prefixe}precisionK_{nom}'] = round(float(y[np.argsort(-s, kind='stable')[:K]].mean()), 4) if K else None
    return res

ref_llm = qualite(cand.p_oui.values, '')
print('\nScore du LLM (départ de prior_llm) :', {k: v for k, v in ref_llm.items() if 'AUC_' in k and 'etendue' not in k})

In [ ]:
# Cellule L2 — Le modèle à flèches apprises (PrereqKT avec poids de flèches appris), et un test rapide
HP_APPRIS = dict(HP, lam_l1=0.01, lam_acyc=0.01, lr_aretes=1e-2, poids_depart=0.1)   # lam_acyc faible : avec le prior LLM, la pénalité de départ est grande

class PrereqKTAppris(GraphKT):
    def __init__(self, n, hp, paires, logits_depart):
        super().__init__(n, hp, np.zeros((n, n), np.float32), indicateurs=True)
        self.register_buffer('src', torch.as_tensor(paires[:, 0]))
        self.register_buffer('dst', torch.as_tensor(paires[:, 1]))
        self.theta = nn.Parameter(torch.as_tensor(logits_depart, dtype=torch.float32))

    def poids(self):
        return torch.sigmoid(self.theta)

    def W(self):                                   # W[a, b] = poids de la flèche a -> b
        W = torch.zeros(self.n, self.n, device=self.theta.device)
        return W.index_put((self.src, self.dst), self.poids())

    def forward(self, q, r):
        W = self.W()
        P_pre = W.T / (W.T.sum(1, keepdim=True) + 1e-6)     # ligne j : prérequis de j (pondérés)
        P_suc = W / (W.sum(1, keepdim=True) + 1e-6)         # ligne i : successeurs de i
        a_pre = 1 - torch.exp(-W.T.sum(1, keepdim=True)); a_suc = 1 - torch.exp(-W.sum(1, keepdim=True))
        H = self.E.weight
        for c in self.couches:
            H = H + torch.relu(c['pre'](P_pre @ H)) * a_pre + torch.relu(c['suc'](P_suc @ H)) * a_suc
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        return self.pred(torch.cat([self.drop(h), H[q[:, 1:]], self.indicateurs_doux(q, r, W)], dim=-1)).squeeze(-1)

    def indicateurs_doux(self, q, r, W):
        # mêmes 6 indicateurs que PrereqKT, mais chaque voisin compte pour le poids de sa flèche
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float(); taux = reussites / essais.clamp(min=1)
        sortie = []
        for M in (W.T, W):
            v = M[q[:, 1:]]
            n_v = v.sum(-1); n_t = (v * tente).sum(-1)
            sortie += [(v * tente * taux).sum(-1) / (n_t + 1e-6), n_t / (n_v + 1e-6), 1 - torch.exp(-n_v)]
        return torch.stack(sortie, -1)

    def acyclicite(self):                          # NOTEARS : tr(exp(W∘W)) − n, nul si et seulement si le graphe est sans cycle
        W = self.W()
        return torch.trace(torch.matrix_exp(W * W)) - self.n

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 0.01, 0.99)
    return np.log(p / (1 - p))

def entrainer_appris(depart, fraction, graine, hp=HP_APPRIS):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    if depart == 'sans_prior':
        l0 = logit(hp['poids_depart']) + np.random.default_rng(graine).normal(0, 0.01, len(PAIRES))
    else:
        l0 = logit(cand.p_oui.values)
    modele = PrereqKTAppris(N, hp, PAIRES, l0).to(DEVICE)
    autres = [p for n_, p in modele.named_parameters() if n_ != 'theta']
    opt = torch.optim.Adam([{'params': autres, 'lr': hp['lr']}, {'params': [modele.theta], 'lr': hp['lr_aretes']}])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float()) \
                + hp['lam_l1'] * modele.poids().mean() + hp['lam_acyc'] * modele.acyclicite()
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    with torch.no_grad():
        poids = modele.poids().cpu().numpy()
        acyc = float(modele.acyclicite())
    return ({'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
             'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1), 'acyclicite': round(acyc, 4),
             'poids_moyen': round(float(poids.mean()), 4), 'fleches_poids_sup_0.5': int((poids > 0.5).sum())}, poids)

# Test rapide : 1 époque à 100 % et à 5 % des élèves (vérifie la mémoire et donne le temps par époque)
for frac in [1.0, 0.05]:
    r_, _ = entrainer_appris('sans_prior', frac, 0, hp=dict(HP_APPRIS, epoques_max=1))
    print(f'{frac:.0%} des élèves : {r_["minutes"]:.2f} min pour 1 époque (évaluation comprise)')
if DEVICE == 'cuda':
    print(f'Mémoire GPU maximale : {torch.cuda.max_memory_allocated() / 1e9:.1f} Go')

In [ ]:
# Cellule L3 — Les entraînements de votre partie, graine par graine (reprise automatique)
PARTIE = 'A'          # 'A' = graines 0, 1, 2 ; 'B' = graines 3, 4 (sur l'autre compte, en même temps)

GRAINES_PARTIE = {'A': [0, 1, 2], 'B': [3, 4]}[PARTIE]
PARTIES = ['A', 'B']
FRACTIONS = [1.0, 0.05]
DOSSIER_POIDS = os.path.join(RESULTATS, 'rq3_poids_appris'); os.makedirs(DOSSIER_POIDS, exist_ok=True)
chemin_l = os.path.join(RESULTATS, f'rq3_aretes_apprises_{PARTIE}.csv')
res_l = pd.read_csv(chemin_l).to_dict('records') if os.path.exists(chemin_l) else []
fait = set()
for p_ in PARTIES:
    c_ = os.path.join(RESULTATS, f'rq3_aretes_apprises_{p_}.csv')
    if os.path.exists(c_):
        fait |= {(x['condition'], float(x['fraction']), int(x['graine'])) for x in pd.read_csv(c_).to_dict('records')}

def noter(ligne):
    res_l.append(ligne); pd.DataFrame(res_l).to_csv(chemin_l, index=False)

t0_ = time.time()
for g_ in GRAINES_PARTIE:
    for frac in FRACTIONS:
        if ('ordre_passage', frac, g_) not in fait:          # référence naïve, sans entraînement
            n2_, na_ = ordre_passage(frac, g_)
            noter({'condition': 'ordre_passage', 'fraction': frac, 'graine': g_, **qualite((na_ + 1) / (n2_ + 2), '')})
        if ('score_llm', frac, g_) not in fait:              # le score du LLM ne dépend pas des données (même valeur partout)
            noter({'condition': 'score_llm', 'fraction': frac, 'graine': g_, **ref_llm})
        if frac == 1.0:                                       # références KT à 100 % (à 5 % : étape 13)
            for lib in ['aucun', 'expert']:
                if (lib, frac, g_) not in fait:
                    res = entrainer('PrereqKT', graphe(lib), fraction=frac, graine=g_)
                    noter({'condition': lib, 'fraction': frac, 'graine': g_, **res})
                    print(f'  graine {g_} | {frac:.0%} | {lib:12s} | AUC 1res = {res["AUC_1re_test"]} | {res["minutes"]} min')
        for depart in ['sans_prior', 'prior_llm']:
            lib = f'appris_{depart}'
            if (lib, frac, g_) in fait:
                continue
            res, poids = entrainer_appris(depart, frac, g_)
            np.save(os.path.join(DOSSIER_POIDS, f'{lib}_{frac}_{g_}.npy'), poids)
            q_ = qualite(poids, '')
            noter({'condition': lib, 'fraction': frac, 'graine': g_, **res, **q_})
            print(f'  graine {g_} | {frac:.0%} | {lib:20s} | AUC 1res = {res["AUC_1re_test"]} | '
                  f'AUC flèches = {q_["AUC_toutes"]} | {res["epoques"]} époques | {res["minutes"]} min')
    print(f'Graine {g_} terminée ({(time.time() - t0_) / 60:.0f} min écoulées)')
print(f'Partie {PARTIE} terminée.')

In [ ]:
# Cellule L4 — Analyse (quand les deux parties sont finies ; marche aussi avec les graines déjà terminées)
#   Après une déconnexion : relancez les cellules 1 à L1, puis celle-ci.
import matplotlib.pyplot as plt
tab = pd.concat([pd.read_csv(os.path.join(RESULTATS, f'rq3_aretes_apprises_{p}.csv')) for p in ['A', 'B']
                 if os.path.exists(os.path.join(RESULTATS, f'rq3_aretes_apprises_{p}.csv'))], ignore_index=True)
tab = tab.drop_duplicates(['condition', 'fraction', 'graine'])
LIBS_A = ['score_llm', 'ordre_passage', 'appris_sans_prior', 'appris_prior_llm']
n_ = tab[tab.condition.isin(LIBS_A)].groupby('graine').size()
graines_ok = sorted(n_[n_ == len(LIBS_A) * 2].index)
tab = tab[tab.graine.isin(graines_ok)]
print(f'Graines complètes analysées : {graines_ok}')

def ic_bootstrap(d, n=10_000, graine=0):
    rng = np.random.default_rng(graine); d = np.asarray(d, dtype=float)
    return np.percentile(rng.choice(d, size=(n, len(d)), replace=True).mean(1), [2.5, 97.5])

# 1) Retrouve-t-on les prérequis experts ?
lignes = []
for frac in [1.0, 0.05]:
    for lib in LIBS_A:
        s = tab[(tab.fraction == frac) & (tab.condition == lib)]
        for sous in SOUS_ENS:
            for mes in ['AUC', 'AUC_etendue', 'precisionK']:
                v = s[f'{mes}_{sous}'].dropna()
                if len(v):
                    lo, hi = ic_bootstrap(v) if v.std() > 0 else (v.mean(), v.mean())
                    lignes.append({'fraction': frac, 'source': lib, 'paires': sous, 'mesure': mes,
                                   'moyenne': round(v.mean(), 4), 'IC95_bas': round(lo, 4), 'IC95_haut': round(hi, 4)})
qual = pd.DataFrame(lignes)
qual.to_csv(os.path.join(RESULTATS, 'rq3_qualite_fleches.csv'), index=False)
print('\n===== AUC face aux flèches expertes (0,5 = hasard) =====')
print(qual[qual.mesure == 'AUC'].pivot_table(index=['fraction', 'source'], columns='paires', values='moyenne').to_string())
base_hasard = {s: round(float(cand.expert.values[m].mean()), 4) for s, m in SOUS_ENS.items()}
print('\nPrécision@K (hasard =', base_hasard, ')')
print(qual[qual.mesure == 'precisionK'].pivot_table(index=['fraction', 'source'], columns='paires', values='moyenne').to_string())

# 2) Hypothèses H5 à H8
hyp = []
for frac in [1.0, 0.05]:
    piv = {c: tab[(tab.fraction == frac) & (tab.condition == c)].set_index('graine') for c in LIBS_A}
    sp = piv['appris_sans_prior']
    tests = [('H5 signal : AUC(sans_prior) > 0,5', sp['AUC_toutes'] - 0.5),
             ("H6 mieux que l'ordre de passage", sp['AUC_toutes'] - piv['ordre_passage']['AUC_toutes']),
             ('H7 les données corrigent le LLM', piv['appris_prior_llm']['AUC_toutes'] - piv['score_llm']['AUC_toutes']),
             ('H8 au-delà de l\'ordre imposé : AUC(sans_prior, ordre variable) > 0,5', sp['AUC_ordre_variable'] - 0.5)]
    for nom, d in tests:
        d = d.dropna()
        hyp.append({'fraction': frac, 'hypothese': nom, 'ecart_moyen': round(d.mean(), 4), 'gagne': f'{int((d > 0).sum())}/{len(d)}',
                    'p': stats.ttest_1samp(d, 0, alternative='greater').pvalue if len(d) > 1 and d.std() > 0 else np.nan})
hyp = pd.DataFrame(hyp)
o = hyp['p'].dropna().sort_values(); m = len(o)
hyp['p_holm'] = (o * (m - np.arange(m))).cummax().clip(upper=1).reindex(hyp.index)
hyp['confirmee_holm_5%'] = hyp['p_holm'] < 0.05
hyp[['p', 'p_holm']] = hyp[['p', 'p_holm']].astype(float).round(4)
hyp.to_csv(os.path.join(RESULTATS, 'rq3_hypotheses.csv'), index=False)
print('\n===== Hypothèses H5 à H8 (Holm sur 8 tests) =====')
print(hyp.to_string(index=False))

# 3) Le graphe appris aide-t-il le KT ? (à 5 %, références de l'étape 13, mêmes graines)
pk = pd.read_csv(os.path.join(RESULTATS, 'rq2_factoriel.csv'))
pk = pk[(pk.fraction == 0.05) & pk.condition.isin(['aucun', 'expert_c100']) & pk.graine.isin(graines_ok)].copy()
pk['condition'] = pk.condition.replace({'expert_c100': 'expert'})
kt = pd.concat([tab[tab.condition.isin(['aucun', 'expert', 'appris_sans_prior', 'appris_prior_llm'])], pk], ignore_index=True)
kt = kt.drop_duplicates(['condition', 'fraction', 'graine'])
kt_tab = kt.groupby(['fraction', 'condition'])[['AUC_1re_test', 'AUC_test']].mean().round(4)
kt_tab.to_csv(os.path.join(RESULTATS, 'rq3_kt.csv'))
print('\n===== KT : AUC des premières tentatives et AUC globale =====')
print(kt_tab.to_string())
cols_g = [c for c in ['acyclicite', 'poids_moyen', 'fleches_poids_sup_0.5'] if c in tab.columns]
print('\nGraphes appris (moyennes) :')
print(tab[tab.condition.str.startswith('appris')].groupby(['fraction', 'condition'])[cols_g].mean().round(3).to_string())

# Figure : AUC des flèches par source et par type de paires
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9), sharey=True)
NOMS = {'score_llm': 'score LLM', 'ordre_passage': 'ordre de passage', 'appris_sans_prior': 'appris (sans prior)',
        'appris_prior_llm': 'appris (prior LLM)'}
for ax, frac in zip(axes, [1.0, 0.05]):
    x = np.arange(len(SOUS_ENS))
    for k, lib in enumerate(LIBS_A):
        s = qual[(qual.fraction == frac) & (qual.source == lib) & (qual.mesure == 'AUC')].set_index('paires').reindex(list(SOUS_ENS))
        err = np.vstack([s.moyenne - s.IC95_bas, s.IC95_haut - s.moyenne]).clip(min=0)
        ax.bar(x + (k - 1.5) * 0.2, s.moyenne, width=0.2, yerr=err, capsize=2, label=NOMS[lib])
    ax.axhline(0.5, ls=':', c='black', lw=1)
    ax.set_xticks(x); ax.set_xticklabels(['toutes les paires', 'ordre imposé', 'ordre variable'])
    ax.set_ylim(0.3, 1.0); ax.set_title(f'{frac:.0%} des élèves')
axes[0].set_ylabel('AUC face aux flèches expertes')
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(RESULTATS, 'rq3_aretes_apprises.png'), dpi=200)
plt.show()

In [ ]:
# Cellule L5 — Rassembler les résultats de l'étape 15 dans un fichier zip et le télécharger
#   À lancer après L4. Si la session a été coupée : relancez d'abord la cellule 1 (et L1 pour le tableau des paires).
import zipfile
if 'cand' in globals():      # tableau des paires par type d'ordre (affiché par L1)
    cand.groupby('ordre').agg(paires=('A', 'size'), dont_expertes=('expert', 'sum'),
                              dont_prerequis_directs_ou_indirects=('ancetre', 'sum')).to_csv(
        os.path.join(RESULTATS, 'rq3_paires_par_ordre.csv'))
FICHIERS = ['rq3_aretes_apprises.png', 'rq3_qualite_fleches.csv', 'rq3_hypotheses.csv', 'rq3_kt.csv',
            'rq3_aretes_apprises_A.csv', 'rq3_aretes_apprises_B.csv', 'rq3_paires_par_ordre.csv']
chemin_zip = '/content/resultats_etape15.zip'
with zipfile.ZipFile(chemin_zip, 'w') as z:
    for f in FICHIERS:
        c = os.path.join(RESULTATS, f)
        if os.path.exists(c):
            z.write(c, arcname=f); print('ajouté   :', f)
        else:
            print('MANQUANT :', f)
try:
    from google.colab import files
    files.download(chemin_zip)
except ImportError:
    print('Hors de Colab : le fichier est ici ->', chemin_zip)